In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import importlib


In [3]:
import copy
import sys
import os

sys.path.append(os.path.abspath("../"))
from campaign_diagram import *
# import campaign_diagram
# campaign_diagram.__file__
# campaign_diagram.CampaignDiagram
## IN this notebook
# guard_position, dual mode, etc....

### Loading Cascade from CSV
One can also load data from a csv.
The fields in the csv should be the same as those in the `Kernel` class.
We introduce new data fields: 
- `fusion_group`: indicates which Einsums should be pipelined together
- `fusion_type`: either NONE (no pipelining) or STRONG (fuse as deep as possible). TODO: WEAK fusion (allows spilling to main memory)
- `windup`: the windup time for this Einsum. It is assumed to be 0 if not present in the CSV.




In [4]:
example_csv = f"./example_cascade_v2.csv"
csv_df = pd.read_csv(example_csv)
csv_df


,Einsum,runtime,Memory_Utilization,Compute_Utilization,mem_latency,comp_latency,total_traffic,Starting_Time,Time_Stamp,End_Time,fusion_group
0,A,2.0,1.0,0.0,2.0,0.0,1.000000e+11,0.0,0.0,2.0,fg1
1,B,3.0,0.0,1.0,0.0,3.0,1.000000e+11,2.0,2.0,5.0,fg1
2,C,1.0,0.4,0.9,0.4,0.9,1.000000e+11,5.0,5.0,6.0,fg2
3,D,1.0,0.9,0.2,0.9,0.2,1.000000e+11,6.0,6.0,7.0,fg2
4,E,1.0,0.1,0.9,0.1,0.9,1.000000e+11,7.0,7.0,8.0,fg3
5,F,1.0,0.4,0.2,0.4,0.2,1.000000e+11,8.0,8.0,9.0,fg3
6,G,2.0,0.6,0.3,1.2,0.6,1.000000e+11,9.0,9.0,11.0,fg3


In [5]:
cascade = load_csv_as_cascade(example_csv, name="Example Cascade")


In [6]:
chart, df = CampaignDiagram(cascade).draw(return_df=True)
chart_dual = CampaignDiagram(cascade).draw(dual_mode=True)
chart.interactive().show()
chart_dual.interactive().show()


~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

Since we want to pipeline by fusion group, we will need to pipeline individually.
If this is fine-grained use `untiled_pipeline_group`:

In [7]:
p1 = cascade.untiled_pipeline_group("fg1", stages=2, spread=True)
p2 = p1.untiled_pipeline_group("fg2", stages=2, spread=True)
p3 = p2.untiled_pipeline_group("fg3", stages=3, spread=True).throttle()
chart, df = CampaignDiagram(p3).draw(return_df=True, include_boundaries=False, \
                                            include_tile_boundaries=False, dual_mode=False)
chart_dual = CampaignDiagram(p3).draw(return_df=False, include_boundaries=False, \
                                            include_tile_boundaries=False, dual_mode=True)
## Fix order of Einsums! I think this is where the bug is coming in?
chart.interactive().show()
chart_dual.interactive().show()

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

We can also indicate wind-up times:


In [8]:
tiled_cascade = cascade
tiled_cascade.set_kernel_windup("A", 0.013)
tiled_cascade.set_kernel_windup("B", 0.013)
tiled_cascade.set_kernel_windup("C", 0.013)
tiled_cascade.set_kernel_windup("D", 0.013)
tiled_cascade.set_kernel_windup("E", 0.013)
tiled_cascade.set_kernel_windup("F", 0.013)
tiled_cascade.set_kernel_windup("G", 0.013)




In [9]:
p1 = tiled_cascade.untiled_pipeline_group("fg1", stages=2, spread=True)
p2 = p1.untiled_pipeline_group("fg2", stages=2, spread=True)
p3 = p2.untiled_pipeline_group("fg3", stages=3, spread=True).throttle()
chart, df = CampaignDiagram(p3).draw(return_df=True, include_boundaries=False, \
                                            include_tile_boundaries=False, dual_mode=False)
chart_dual = CampaignDiagram(p3).draw(return_df=False, include_boundaries=False, \
                                            include_tile_boundaries=False, dual_mode=True)
## Fix order of Einsums! I think this is where the bug is coming in?
chart.interactive().show()
chart_dual.interactive().show()

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

### Auto Pipeline from CSV data

In [10]:
cascade = load_csv_as_cascade("pedagogical_einsum_dataset.csv", name="Pedagogical Complex Cascade")

df = pd.read_csv("pedagogical_einsum_dataset.csv")
df.head(20) 

,Einsum,fusion_group,runtime,mem_latency,comp_latency,Memory_Utilization,Compute_Utilization,total_traffic,Starting_Time,Time_Stamp,End_Time
0,A,fg1,0.20,0.126640,0.073360,1.00,0.75,1.000000e+09,0.00,0.00,0.20
1,B,fg1,0.02,0.015263,0.004737,0.00,0.25,0.000000e+00,0.20,0.20,0.22
2,C,fg1,0.05,0.010023,0.039977,0.25,0.75,1.000000e+11,0.22,0.22,0.27
3,D,fg1,0.20,0.159065,0.040935,0.75,0.25,1.000000e+11,0.27,0.27,0.47
4,E,fg2,0.01,0.005705,0.004295,0.00,0.25,1.000000e+11,0.47,0.47,0.48
5,F,fg2,0.05,0.028350,0.021650,0.75,0.75,5.000000e+11,0.48,0.48,0.53
6,G,fg2,0.05,0.010212,0.039788,0.25,1.00,0.000000e+00,0.53,0.53,0.58
7,H,fg2,0.20,0.042767,0.157233,0.25,0.25,5.000000e+11,0.58,0.58,0.78
8,I,fg3,0.20,0.102973,0.097027,0.00,0.25,5.000000e+11,0.78,0.78,0.98
9,J,fg3,0.01,0.004399,0.005601,0.25,0.75,0.000000e+00,0.98,0.98,0.99


In [11]:
chart, df = CampaignDiagram(cascade).draw(return_df=True)
chart_dual = CampaignDiagram(cascade).draw(dual_mode=True)

chart.display()
chart_dual.display()
df

alt.LayerChart(...)

alt.LayerChart(...)

,Einsum,fusion_group,fusion_type,Starting_Time,Time_Stamp,End_Time,cb_End_Time,mb_End_Time,full_End_Time,runtime,...,compute_color,bw_color,throttled,throttled_duration,mem_throttled_duration,comp_throttled_duration,comp_perc,bw_perc,compute_type,memory_type
0,A,fg1,STRONG,0.00,0.00,0.2000,0.1500,0.2000,0.20,0.20,...,#332288,#9990C3,False,0.0000,0.0000,0.0500,1.0,1.0,Compute Pool,Memory Pool
1,B,fg1,STRONG,0.20,0.20,0.2050,0.2050,0.2000,0.22,0.02,...,#117733,#88BB99,True,0.0150,0.0200,0.0150,1.0,1.0,Compute Pool,Memory Pool
2,C,fg1,STRONG,0.22,0.22,0.2575,0.2575,0.2325,0.27,0.05,...,#882255,#C390AA,True,0.0125,0.0375,0.0125,1.0,1.0,Compute Pool,Memory Pool
3,D,fg1,STRONG,0.27,0.27,0.4200,0.3200,0.4200,0.47,0.20,...,#44AA99,#A1D4CC,True,0.0500,0.0500,0.1500,1.0,1.0,Compute Pool,Memory Pool
4,E,fg2,STRONG,0.47,0.47,0.4725,0.4725,0.4700,0.48,0.01,...,#999933,#CCCC99,True,0.0075,0.0100,0.0075,1.0,1.0,Compute Pool,Memory Pool
5,F,fg2,STRONG,0.48,0.48,0.5175,0.5175,0.5175,0.53,0.05,...,#AA4499,#D4A1CC,True,0.0125,0.0125,0.0125,1.0,1.0,Compute Pool,Memory Pool
6,G,fg2,STRONG,0.53,0.53,0.5800,0.5800,0.5425,0.58,0.05,...,#DDCC77,#EEE5BB,False,0.0000,0.0375,0.0000,1.0,1.0,Compute Pool,Memory Pool
7,H,fg2,STRONG,0.58,0.58,0.6300,0.6300,0.6300,0.78,0.20,...,#88CCEE,#C3E5F6,True,0.1500,0.1500,0.1500,1.0,1.0,Compute Pool,Memory Pool
8,I,fg3,STRONG,0.78,0.78,0.8300,0.8300,0.7800,0.98,0.20,...,#CC6677,#E5B2BB,True,0.1500,0.2000,0.1500,1.0,1.0,Compute Pool,Memory Pool
9,J,fg3,STRONG,0.98,0.98,0.9875,0.9875,0.9825,0.99,0.01,...,#661100,#B2887F,True,0.0025,0.0075,0.0025,1.0,1.0,Compute Pool,Memory Pool


### Now, we want to pipeline! 
Call `auto_pipline_by_group_size` to automatically pipeline each group of Einsums based on their fusion group size.
- pass the `df` returned by the CampaignDiagram class to the funcion, as well as any chart arguments

    

In [12]:
chart, pipeline_df = auto_pipeline_chart_by_group_size(
    df,
    spread=True,
    chart_kwargs={
        "return_df": True, 
        "include_boundaries": False,
        "include_tile_boundaries": False,
        "dual_mode": False,
        "title": "Complex Pipelined Cascade"
    },
)

chart.display()
pipeline_df

alt.LayerChart(...)

,Einsum,fusion_group,fusion_type,Starting_Time,Time_Stamp,End_Time,cb_End_Time,mb_End_Time,full_End_Time,runtime,...,compute_color,bw_color,throttled,throttled_duration,mem_throttled_duration,comp_throttled_duration,comp_perc,bw_perc,compute_type,memory_type
0,A,fg1,NONE,0.0000,0.0000,0.362500,0.271875,0.362500,0.3625,0.3625,...,#332288,#9990C3,False,0.000000,0.000000,0.090625,1.0,1.0,Compute Pool,Memory Pool
1,D,fg1,NONE,0.0000,0.0000,0.271875,0.090625,0.271875,0.3625,0.3625,...,#44AA99,#A1D4CC,True,0.090625,0.090625,0.271875,1.0,1.0,Compute Pool,Memory Pool
2,C,fg1,NONE,0.0000,0.0000,0.271875,0.271875,0.090625,0.3625,0.3625,...,#882255,#C390AA,True,0.090625,0.271875,0.090625,1.0,1.0,Compute Pool,Memory Pool
3,B,fg1,NONE,0.0000,0.0000,0.090625,0.090625,0.000000,0.3625,0.3625,...,#117733,#88BB99,True,0.271875,0.362500,0.271875,1.0,1.0,Compute Pool,Memory Pool
4,H,fg2,NONE,0.3625,0.3625,0.412500,0.412500,0.412500,0.5625,0.2000,...,#88CCEE,#C3E5F6,True,0.150000,0.150000,0.150000,1.0,1.0,Compute Pool,Memory Pool
5,F,fg2,NONE,0.3625,0.3625,0.512500,0.512500,0.512500,0.5625,0.2000,...,#AA4499,#D4A1CC,True,0.050000,0.050000,0.050000,1.0,1.0,Compute Pool,Memory Pool
6,G,fg2,NONE,0.3625,0.3625,0.562500,0.562500,0.412500,0.5625,0.2000,...,#DDCC77,#EEE5BB,False,0.000000,0.150000,0.000000,1.0,1.0,Compute Pool,Memory Pool
7,E,fg2,NONE,0.3625,0.3625,0.412500,0.412500,0.362500,0.5625,0.2000,...,#999933,#CCCC99,True,0.150000,0.200000,0.150000,1.0,1.0,Compute Pool,Memory Pool
8,K,fg3,NONE,0.5625,0.5625,0.765000,0.613125,0.765000,0.7650,0.2025,...,#6699CC,#B2CCE5,False,0.000000,0.000000,0.151875,1.0,1.0,Compute Pool,Memory Pool
9,J,fg3,NONE,0.5625,0.5625,0.714375,0.714375,0.613125,0.7650,0.2025,...,#661100,#B2887F,True,0.050625,0.151875,0.050625,1.0,1.0,Compute Pool,Memory Pool
